In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)


class State(TypedDict):
    topic: str
    technical: str
    business: str
    summary: str


def technical_analysis(state: State):
    response = llm.invoke(
        f"Give a technical analysis of {state['topic']}"
    )

    return {
        "technical": response.content
    }


def business_analysis(state: State):
    response = llm.invoke(
        f"Give a business analysis of {state['topic']}"
    )

    return {
        "business": response.content
    }


def summarize(state: State):
    response = llm.invoke(
        f"""
        Create a final summary.

        Technical Analysis:
        {state['technical']}

        Business Analysis:
        {state['business']}
        """
    )

    return {
        "summary": response.content
    }


# Create graph
graph = StateGraph(State)

# Add nodes
graph.add_node("technical", technical_analysis)
graph.add_node("business", business_analysis)
graph.add_node("summarize", summarize)

# Parallel branches
graph.add_edge(START, "technical")
graph.add_edge(START, "business")

# Both branches converge here
graph.add_edge("technical", "summarize")
graph.add_edge("business", "summarize")

graph.add_edge("summarize", END)

app = graph.compile()

result = app.invoke({
    "topic": "Generative AI",
    "technical": "",
    "business": "",
    "summary": ""
})

print(result["summary"])